### Importing Packages

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings('ignore')

### Creating Lags

In [2]:
data = {'sales': [100, 120, 130, 150, 170, 200, 220]}
df = pd.DataFrame(data)

# Creating lag features
df['lag1'] = df['sales'].shift(1)
df['lag2'] = df['sales'].shift(2)
df.dropna(inplace=True)
print("Output\n")
print(df)

Output

   sales   lag1   lag2
2    130  120.0  100.0
3    150  130.0  120.0
4    170  150.0  130.0
5    200  170.0  150.0
6    220  200.0  170.0


### Linear Regression

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Features and target
X = df[['lag1', 'lag2']]
y = df['sales']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

# Train Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))


MSE: 55.5555555555553


### Decision Tree Regressor

In [4]:
from sklearn.tree import DecisionTreeRegressor

tree_model = DecisionTreeRegressor(max_depth=3)
tree_model.fit(X_train, y_train)
y_tree_pred = tree_model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_tree_pred))

MSE: 1700.0


### Random Forest Regressor

In [5]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_rf_pred = rf_model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_rf_pred))

MSE: 2384.840000000001


### Adding a rolling mean

In [6]:
df['rolling_mean'] = df['sales'].rolling(window=2).mean().shift(1)

In [7]:
df.head()

,sales,lag1,lag2,rolling_mean
2,130,120.0,100.0,NaN
3,150,130.0,120.0,NaN
4,170,150.0,130.0,140.0
5,200,170.0,150.0,160.0
6,220,200.0,170.0,185.0


### Stacked Prediction

In [8]:
import numpy as np

# Stack predictions
stacked_pred = (y_pred + y_rf_pred) / 2
print("Stacked MSE:", mean_squared_error(y_test, stacked_pred))

Stacked MSE: 497.09888888888963


### Time Series Split

In [9]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=3)
for train_index, test_index in tscv.split(X):
    print("Train:", train_index, "Test:", test_index)

Train: [0 1] Test: [2]
Train: [0 1 2] Test: [3]
Train: [0 1 2 3] Test: [4]
